# Day 6: MFC Hierarchy — Nations, Firms, and Individual Traders

*Alpha Flow Research · HongJin HE · July 2026*

---

## Three Levels of the Market Game

Financial markets are not flat. They have a **strict hierarchy** of decision-makers, each operating at different timescales with different objectives and information sets:

| Level | Players | Timescale | Objective | Game Type |
|---|---|---|---|---|
| **L1** | Central banks, sovereign funds | Years | Social welfare / macro stability | **MFC** (cooperative) |
| **L2** | Institutional investors, hedge funds | Months | AUM growth, Sharpe | **MFG** (competitive) |
| **L3** | Retail traders, HFT | Seconds-days | P&L | **MFG** (competitive) |

The key distinction:
- **Mean-Field Game (MFG)**: Each player optimizes *against* the field → Nash equilibrium
- **Mean-Field Control (MFC)**: Central planner optimizes *for* the field → social optimum

Central banks are the canonical MFC player: the Fed sets rates to optimize the entire economy, not to beat other central banks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import minimize

np.random.seed(2026)

# ── Illustrate the Price of Anarchy: MFG vs MFC ──────────────────────────
# A simple N-player game: each player allocates α_i to a risky asset.
# Price impact: r(α_1,...,α_N) = μ - λ·mean(α_i) + σ·ε
# Individual objective: max E[α_i · r] - γ/2 · α_i²

N_players = [10, 50, 200, 1000]
mu_base, lambda_impact, sigma, gamma = 0.10, 0.5, 0.15, 2.0

mfg_welfare = []  # Nash equilibrium welfare per player
mfc_welfare = []  # Social optimum welfare per player

for N in N_players:
    # MFG Nash: each player takes ā = mean as given → α* = μ / (γ + λ/N) ... wait
    # In MFG limit (N→∞), α* solves: μ - λ·α* - γ·α* = 0 → α*_MFG = μ/(γ+λ)
    alpha_mfg = mu_base / (gamma + lambda_impact)
    welfare_mfg = alpha_mfg * (mu_base - lambda_impact * alpha_mfg) - gamma/2 * alpha_mfg**2

    # MFC Social optimum: planner internalizes impact → α* solves d/dα[N·W] = 0
    # Planner's α: μ - 2λ·α - γ·α = 0 → α*_MFC = μ/(γ+2λ)
    alpha_mfc = mu_base / (gamma + 2*lambda_impact)
    welfare_mfc = alpha_mfc * (mu_base - lambda_impact * alpha_mfc) - gamma/2 * alpha_mfc**2

    mfg_welfare.append(welfare_mfg)
    mfc_welfare.append(welfare_mfc)

print('Price of Anarchy (MFC welfare / MFG welfare):')
for N, w_mfg, w_mfc in zip(N_players, mfg_welfare, mfc_welfare):
    poa = w_mfc / w_mfg if w_mfg > 0 else float('inf')
    print(f'  N={N:4d}: α_MFG={mu_base/(gamma+lambda_impact):.3f}, α_MFC={mu_base/(gamma+2*lambda_impact):.3f}, PoA={poa:.3f}')

## The Bi-Level Optimization

The full hierarchy gives a **bi-level MFG/MFC** problem:

**Upper level (L1 — Central Banks, MFC)**:
$$\min_{\pi^{\text{CB}}} \mathbb{E}\left[\int_0^T \mathcal{L}^{\text{macro}}(z_t, \pi^{\text{CB}}_t) dt\right]$$

subject to: the lower level Nash equilibrium

**Lower level (L2/L3 — Funds/Traders, MFG)**:
$$V^i(t, z) = \sup_{\pi^i} \mathbb{E}\left[\int_t^T f^i(z_s, \pi^i_s, \mu_s) ds + g^i(z_T)\right]$$

where $\mu_t$ is the law of all L2/L3 agents (the mean-field).

This creates a **Stackelberg game structure** — L1 leads, L2/L3 follow.

In [ ]:
# Visualize the hierarchy and information flows
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

# Left: Hierarchy pyramid
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)
ax1.axis('off')

levels = [
    (5, 8.5, 'L1: Central Banks\n(MFC — Cooperative)', 'darkred', 2.0),
    (5, 5.5, 'L2: Hedge Funds / Institutions\n(MFG — Competitive)', 'steelblue', 4.0),
    (5, 2.5, 'L3: Retail / HFT\n(MFG — Competitive)', 'green', 6.5),
]

for x, y, label, color, width in levels:
    rect = plt.Rectangle((x-width/2, y-0.8), width, 1.6, 
                          color=color, alpha=0.7, zorder=2)
    ax1.add_patch(rect)
    ax1.text(x, y, label, ha='center', va='center', color='white', 
             fontsize=9, fontweight='bold', zorder=3)

# Arrows: top-down policy, bottom-up price signal
for y_start, y_end in [(7.7, 6.3), (4.7, 3.3)]:
    ax1.annotate('', xy=(3.5, y_end), xytext=(3.5, y_start),
                arrowprops=dict(arrowstyle='->', color='orange', lw=2))
    ax1.annotate('', xy=(6.5, y_start), xytext=(6.5, y_end),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))

ax1.text(2.8, 7.0, 'policy', ha='center', fontsize=8, color='orange')
ax1.text(7.2, 7.0, 'prices', ha='center', fontsize=8, color='gray')
ax1.text(2.8, 4.0, 'capital', ha='center', fontsize=8, color='orange')
ax1.text(7.2, 4.0, 'vol signal', ha='center', fontsize=8, color='gray')
ax1.set_title('Three-Level Market Hierarchy\n(MFC at top, MFG below)', fontsize=11)

# Right: Timescale separation
timescales = {'L1\n(CB)': (100, 5, 'darkred'), 
              'L2\n(HF)': (20, 2, 'steelblue'),
              'L3\n(Retail)': (1, 0.5, 'green'),
              'L3\n(HFT)': (0.001, 0.1, 'purple')}

for label, (timescale_days, capital_bn, color) in timescales.items():
    ax2.scatter(timescale_days, capital_bn, s=300, color=color, zorder=3, alpha=0.8)
    ax2.annotate(label, (timescale_days, capital_bn), 
                textcoords='offset points', xytext=(8,5), fontsize=9)

ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('Characteristic timescale (days)')
ax2.set_ylabel('Capital per player (\$B, schematic)')
ax2.set_title('Timescale vs Capital: Clear Separation\nEnables Hierarchical Modeling', fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/mfc_hierarchy_detail.png', dpi=150, bbox_inches='tight')
plt.show()

## Why Timescale Separation Matters

The hierarchy is tractable because of **timescale separation**: L1 moves on yearly cycles, L2 on quarterly, L3 on daily-secondly. This means:

1. **L1 parameters are approximately fixed** when solving the L2/L3 Nash equilibrium
2. **L2/L3 Nash provides the state distribution** that L1 optimizes over
3. We can solve each level sequentially via **backward induction**

Mathematically, this is the **singular perturbation** structure:
$$\epsilon \cdot dX^{\text{slow}}_t = b^{\text{slow}} dt, \quad dX^{\text{fast}}_t = b^{\text{fast}} dt + \sigma dW_t$$

As $\epsilon \to 0$, the fast variable reaches its invariant measure before the slow variable moves appreciably.

**Implication for implementation**: the Game Module G only needs to solve the L2/L3 Nash equilibrium. L1 policy (macro regime) enters as an **exogenous parameter** estimated from the Encoder's latent dimension $z^{(\text{macro})}$.

This is why our controller remains tractable even with thousands of agents — see Day 5 for the MFG formulation.